# The paper's ResNet-56 models in losslens

This notebook runs losslens on the **pretrained models published with the original paper**
(Li et al., *Visualizing the Loss Landscape of Neural Nets*, NeurIPS 2018), so we can compare
losslens' plots with the paper's figures:

- **ResNet-56**, with skip connections;
- **ResNet-56-noshort**, the same network without skip connections.

The paper's main finding for these two models: with skip connections the landscape is a smooth,
nearly convex basin; without them it becomes chaotic, with sharp walls and many local minima.

The settings below match the original code's settings for these figures:

| Original `plot_surface.py` | losslens |
|---|---|
| `--x=-1:1:51 --y=-1:1:51` | `scale=1.0, resolution=51` |
| `--xnorm filter --ynorm filter` | always (filter normalization) |
| `--xignore biasbn --yignore biasbn` | always (biases and BatchNorm not moved) |
| `--dir_type weights` | always |
| full training set (50,000 images) | `n_samples` (2,000 by default, see below) |

Run it on a GPU runtime. Computing both surfaces at the paper's resolution takes a while
(losslens prints an estimate); surfaces are saved to disk, so re-running the notebook reuses them.

## 1. Setup

In [ ]:
%pip install -q git+https://github.com/iturbideM/losslens datasets gdown
# The model architectures come from the original code (the fork).
![ -d loss-landscape ] || git clone -q https://github.com/iturbideM/loss-landscape

In [ ]:
import glob
import os
import re
import sys
import tarfile

import gdown
import numpy as np
import torch
import torch.nn as nn
from datasets import load_dataset
from IPython.display import display
from torch.utils.data import DataLoader, TensorDataset

import losslens

sys.path.insert(0, "loss-landscape")
from cifar10.models.resnet import ResNet56, ResNet56_noshort

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 2. Download the pretrained models

The authors shared the checkpoints on Google Drive (links in the original README). If the download
fails (Google Drive sometimes limits downloads of shared files), download the two `.tar.gz` files
by hand from the original README and put them in a `pretrained/` folder next to this notebook.

In [ ]:
CHECKPOINTS = {
    # name: (Google Drive file id, archive name, architecture)
    "ResNet-56": ("12oxkvfaKcPyyHiOevVNTBzaQ1zAFlNPX", "resnet56.tar.gz", ResNet56),
    "ResNet-56-noshort": ("1eUvYy3HaiCVHTzi3MHEZGgrGOPACLMkR", "resnet56_noshort.tar.gz", ResNet56_noshort),
}

def extract(archive, folder):
    with tarfile.open(archive) as tar:
        try:
            tar.extractall(folder, filter="data")
        except TypeError:  # Python without extraction filters
            tar.extractall(folder)

def find_checkpoint(folder):
    """The model_<epoch>.t7 file with the highest epoch (the final model)."""
    files = glob.glob(os.path.join(folder, "**", "*.t7"), recursive=True)
    models = [f for f in files if re.search(r"model_\d+\.t7$", f)] or files
    if not models:
        raise FileNotFoundError(f"No .t7 checkpoint found in {folder}")
    return max(models, key=lambda f: int(m.group(1)) if (m := re.search(r"model_(\d+)\.t7$", f)) else -1)

os.makedirs("pretrained", exist_ok=True)
checkpoint_files = {}
for name, (file_id, archive, _) in CHECKPOINTS.items():
    archive_path = os.path.join("pretrained", archive)
    if not os.path.exists(archive_path):
        gdown.download(id=file_id, output=archive_path, quiet=False)
    folder = os.path.join("pretrained", name)
    if not os.path.isdir(folder):
        extract(archive_path, folder)
    checkpoint_files[name] = find_checkpoint(folder)
    print(f"{name}: {checkpoint_files[name]}")

## 3. Load the models

The checkpoints were saved with PyTorch 0.4 as a dictionary holding the weights (`state_dict`)
and the test accuracy reached during training (`acc`).

In [ ]:
def load_pretrained(architecture, path):
    try:
        stored = torch.load(path, map_location="cpu", weights_only=True)
    except Exception:
        # Old checkpoints sometimes need the full unpickler. Only do this for files you trust.
        stored = torch.load(path, map_location="cpu", weights_only=False)
    state = stored.get("state_dict", stored)
    state = {k.removeprefix("module."): v for k, v in state.items()}
    net = architecture()
    net.load_state_dict(state)
    return net.eval(), stored.get("acc")

models = {}
for name, (_, _, architecture) in CHECKPOINTS.items():
    models[name], recorded_acc = load_pretrained(architecture, checkpoint_files[name])
    n_params = sum(p.numel() for p in models[name].parameters())
    print(f"{name}: {n_params:,} parameters, test accuracy recorded in checkpoint: {recorded_acc}")

## 4. Data

CIFAR-10 from the Hugging Face Hub, normalized exactly like the original code (no augmentation).
The loss landscape is computed on the **training** set, like in the paper.

In [ ]:
def load_cifar10(split):
    ds = load_dataset("uoft-cs/cifar10", split=split)
    X = np.stack([np.asarray(img) for img in ds["img"]])        # (N, 32, 32, 3) uint8
    return torch.from_numpy(X).permute(0, 3, 1, 2).contiguous(), torch.tensor(ds["label"])

# Normalization used by the original code (cifar10/dataloader.py)
MEAN = torch.tensor([125.3, 123.0, 113.9]).view(1, 3, 1, 1) / 255
STD = torch.tensor([63.0, 62.1, 66.7]).view(1, 3, 1, 1) / 255

def normalize(x):
    return (x.float() / 255 - MEAN) / STD

X_train, y_train = load_cifar10("train")
X_test, y_test = load_cifar10("test")
X_train, X_test = normalize(X_train), normalize(X_test)

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=256)
print("train:", tuple(X_train.shape), " test:", tuple(X_test.shape))

### Sanity check: did the weights load correctly?

If the architecture, the weights or the normalization were wrong, the accuracy would be close to
chance (10%). It should be close to the accuracy recorded in each checkpoint.

In [ ]:
@torch.no_grad()
def accuracy(net, X, y, batch_size=500):
    net = net.to(device).eval()
    correct = 0
    for i in range(0, len(X), batch_size):
        correct += (net(X[i:i + batch_size].to(device)).argmax(1).cpu() == y[i:i + batch_size]).sum().item()
    return 100 * correct / len(X)

for name, net in models.items():
    print(f"{name}: test accuracy {accuracy(net, X_test, y_test):.2f}%")

## 5. Compute the landscapes

`RESOLUTION = 51` matches the paper. For a quick first look, set it to 25 (about 4 times faster).

The paper averaged the loss over all 50,000 training images. losslens uses a fixed random subset;
2,000 images give the same shape at a fraction of the cost. Raise `N_SAMPLES` for a closer match
of the exact values.

Each surface is saved to a file named after its settings, so re-running reuses it.

In [ ]:
SCALE = 1.0
RESOLUTION = 51
N_SAMPLES = 2000

surfaces = {}
for name, net in models.items():
    path = f"{name}_scale{SCALE}_res{RESOLUTION}_n{N_SAMPLES}.npz"
    if os.path.exists(path):
        surfaces[name] = losslens.load(path)
        print(f"{name}: loaded {path}")
        continue
    print(f"{name}:")
    surfaces[name] = losslens.surface(net, train_loader, loss_fn=nn.CrossEntropyLoss(),
                                      scale=SCALE, resolution=RESOLUTION, n_samples=N_SAMPLES)
    surfaces[name].save(path)

## 6. 2D contours: losslens vs. the paper

The paper's contour plot for ResNet-56, made by the original code with the same settings
(color scale up to a loss of 10):

![Paper: ResNet-56 2D contour](https://raw.githubusercontent.com/iturbideM/loss-landscape/master/doc/images/resnet56_sgd_lr%3D0.1_bs%3D128_wd%3D0.0005/model_300.t7_weights_xignore%3Dbiasbn_xnorm%3Dfilter_yignore%3Dbiasbn_ynorm%3Dfilter.h5_%5B-1.0%2C1.0%2C51%5Dx%5B-1.0%2C1.0%2C51%5D.h5_train_loss_2dcontour.jpg)

losslens, with the color scale also capped at 10 (`zmax=10`) to match:

In [ ]:
surfaces["ResNet-56"].plot(zmax=10, title="losslens: ResNet-56")

Both models side by side, on one shared color scale, with their numbers:

In [ ]:
losslens.compare(surfaces, zmax=10)

## 7. 3D surfaces: losslens vs. the paper

The paper's 3D renders (made with ParaView, log scale, loss capped at 10). Left: ResNet-56-noshort.
Right: ResNet-56.

<img src="https://raw.githubusercontent.com/iturbideM/loss-landscape/master/doc/images/resnet56_noshort_small.jpg" width="45%"> <img src="https://raw.githubusercontent.com/iturbideM/loss-landscape/master/doc/images/resnet56_small.jpg" width="45%">

losslens, with the same log scale and cap:

In [ ]:
surfaces["ResNet-56-noshort"].plot_3d(log=True, zmax=10, title="losslens: ResNet-56-noshort")

In [ ]:
surfaces["ResNet-56"].plot_3d(log=True, zmax=10, title="losslens: ResNet-56")

## 8. Slices

In [ ]:
for name, surf in surfaces.items():
    display(surf.plot_slice(zmax=10, title=name))

## What to expect, and why it won't match exactly

The plots should agree with the paper **qualitatively**: a smooth, round basin for ResNet-56 and a
much more irregular, sharper landscape for ResNet-56-noshort. They will not match contour for
contour, because:

- **The directions are random.** losslens and the original code draw different random directions,
  so we are looking at a different 2D slice of the same model. Changing `seed` here would change
  the details too.
- **The data differs.** The paper averages over all 50,000 training images, losslens over a subset
  (`N_SAMPLES`). This mostly affects exact values, not the shape.

A good check that the shape is a property of the model and not of one slice: recompute with
`seed=1` and see whether the picture stays the same.